In [0]:
#Đọc dữ liệu và Gắn nhãn Cảnh báo học vụ

import warnings
warnings.filterwarnings("ignore")

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from mlflow.models import infer_signature
import mlflow

spark = SparkSession.builder.getOrCreate()
mlflow.set_registry_uri("databricks-uc")

df_raw = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/Volumes/workspace/smart_gpa/dataset/diem_sinh_vien.csv")

# Mã hóa loại học phần
indexer = StringIndexer(inputCol="loai_hoc_phan", outputCol="loai_hoc_phan_encoded")
df_processed = indexer.fit(df_raw).transform(df_raw)

# GẮN NHÃN CẢNH BÁO HỌC VỤ: Nếu status_canh_bao_final chứa chữ 'CANH BAO' hoặc 'Nguy co' -> Nhãn = 1.0, an toàn -> 0.0
df_ml_warning = df_processed.withColumn(
    "label",
    when(col("status_canh_bao_final") != "An toan", 1.0).otherwise(0.0)
)

# Chọn tính năng đầu vào
feature_cols = ["diem_tich_luy_hien_tai", "diem_trung_binh_thuc_hanh", "tong_so_chi", "loai_hoc_phan_encoded"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

df_final_warning = assembler.transform(df_ml_warning).select("features", "label")

In [0]:
#Huấn luyện và Đăng ký mô hình Cảnh báo học vụ

train_data, test_data = df_final_warning.randomSplit([0.8, 0.2], seed=789)

rf_warning = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=60, maxDepth=5, seed=789)

with mlflow.start_run(run_name="Predict_Academic_Warning") as run:
    model_warning = rf_warning.fit(train_data)
    predictions = model_warning.transform(test_data)
    
    evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
    accuracy = evaluator.evaluate(predictions)
    print(f"Độ chính xác (Accuracy) mô hình Cảnh báo học vụ: {round(accuracy * 100, 2)}%")
    
    X_train = train_data.select("features")
    y_pred = predictions.select("prediction")
    signature = infer_signature(X_train, y_pred)
    
    mlflow.spark.log_model(
        spark_model=model_warning,
        artifact_path="model",
        signature=signature,
        dfs_tmpdir="/Volumes/workspace/smart_gpa/dataset/mlflow_tmp_warning"
    )
    run_id_warning = run.info.run_id

# Đăng ký mô hình thứ hai lên Unity Catalog
full_model_name_warning = "workspace.smartgpa_db.academic_warning_model"
model_details_warning = mlflow.register_model(model_uri=f"runs:/{run_id_warning}/model", name=full_model_name_warning)
print(f"Mô hình Cảnh báo học vụ đã đăng ký thành công phiên bản: Version {model_details_warning.version}")

Độ chính xác (Accuracy) mô hình Cảnh báo học vụ: 100.0%


2026/05/31 14:04:49 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.1.0+databricks.connect.18.0.6) contains a local version label (+databricks.connect.18.0.6). MLflow logged a pip requirement for this package as 'pyspark==4.1.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/05/31 14:04:53 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-7c878e4c-a46a-4840-997c-07/tmptx6yczvu/model, flavor: spark). Fall back to return ['pyspark==4.1.0']. Set logging level to DEBUG to see the full traceback. 
Registered model 'workspace.smartgpa_db.academic_warning_model' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/24 [00:00<?, ?it/s]

🔗 Created version '2' of model 'workspace.smartgpa_db.academic_warning_model': https://dbc-f659bab4-1de9.cloud.databricks.com/explore/data/models/workspace/smartgpa_db/academic_warning_model/version/2?o=7474656584148137


Mô hình Cảnh báo học vụ đã đăng ký thành công phiên bản: Version 2


In [0]:
# 